## Data Understanding and Cleaning: ‘Bot vs Human’ 
Group 14

Students: Romāns Jurčonoks(3848191), Jan-Jaap Lankhaar(3872319), Amirmohammad Masoumi(3648672), Meraj Mosaybezadeh(37592)

## Inspect the dataset
This is the part where we explored given dataset and prepared data for further analysis. 

To start working with the dataset in python, we need to import required libraries and functions(#1).
To work with the given dataset, first we need to load a dataset from the CSV file and reading it through pd.read_csv() (#2).
At step (#3) we are using dataset.shape and dataset.columns to figure out number of rows,columns and inspect dataset column names and preview of the data. 

In [217]:
#1
import pandas as pd
import pandasql as ps
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from math import sqrt
from joblib import dump, load

In [218]:
#2
dataset_path = "data-7.csv"
dataset = pd.read_csv(dataset_path)

In [219]:
#3
print("Number of rows and columns:", dataset.shape)
print("\nColumn names:")
for col in dataset.columns:
    print(" -", col)
print("\nPreview of the data:")
dataset

Number of rows and columns: (400, 20)

Column names:
 - clicks
 - median_click_interval_ms
 - click_interval_std_ms
 - mouse_path_curvature
 - mouse_distance_px
 - idle_ratio
 - scroll_entropy
 - digraph_latency_ms
 - dwell_field1_ms
 - dwell_field2_ms
 - dwell_field3_ms
 - session_duration_ms
 - pages_visited
 - timezone_mismatch
 - referrer_type
 - webdriver_flag
 - button_hover_ms
 - copy_events
 - viewport_width
 - bot

Preview of the data:


,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,dwell_field1_ms,dwell_field2_ms,dwell_field3_ms,session_duration_ms,pages_visited,timezone_mismatch,referrer_type,webdriver_flag,button_hover_ms,copy_events,viewport_width,bot
0,30,226,209,0.536,10735,0.654,3.05,151,758,469,250,10745,9,0,search,0,97,0,414,0
1,32,410,24,0.451,55982,0.945,2.17,128,923,935,941,22558,4,0,search,0,136,1,414,1
2,31,839,49,0.200,2867,0.809,1.36,71,526,536,522,43276,6,0,ad,0,84,2,375,1
3,32,318,75,0.731,8259,0.525,1.65,86,786,725,468,15483,3,0,direct,0,263,1,1440,0
4,28,666,517,0.415,51596,0.097,2.61,101,705,429,672,25198,8,0,ad,1,145,2,1366,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,26,212,200,0.439,11616,0.338,2.43,1,0,694,405,4759,4,0,social,0,137,1,375,0
396,35,448,269,0.844,6217,0.263,1.46,45,498,953,564,19235,8,1,direct,0,141,3,1440,0
397,30,434,145,0.734,9269,0.178,2.88,101,653,496,480,16872,8,0,social,0,122,2,1366,0
398,30,522,160,0.856,2106,0.204,1.22,169,1011,205,570,16304,5,0,search,0,0,1,1920,0


## Cleaning step
The cleaning decision we made was imputation. 

The idea is to replace outliers with the average value. We set the boundaries "lower" and "upper", if the value in the column is bigger than upper or lower boundary, than it is considered to be outlier. The variable mean_value computes the mean value to replace outliers that contain bigger or smaller values than boundaries. 

In [220]:
numerical_cols = dataset.select_dtypes(include="number").columns.drop("bot", errors="ignore") 

dataset[numerical_cols] = dataset[numerical_cols].astype(float)

for col in numerical_cols:
    lower = dataset[col].quantile(0.04)
    upper = dataset[col].quantile(0.96)
    mean_value = dataset[col].mean()

    dataset.loc[
        (dataset[col] < lower) | (dataset[col] > upper),
        col
    ] = mean_value

## Exploring the Dataset Using SQL Queries
To understand which features might actually affect final result and which do not, we created SQL queries for each feature.

In each query, we selected the bot column and calculated the average value of the feature to see whether there is a noticeable difference between bots and humans.

In [221]:
# function query(sql) will help us write our SQL queries easily
def query(sql):
    return ps.sqldf(sql, globals())

In [222]:
query("""
    SELECT bot, AVG(mouse_path_curvature) FROM dataset GROUP BY bot
""")

,bot,AVG(mouse_path_curvature)
0,0,0.567081
1,1,0.353012


In [223]:
query (""" 
SELECT bot, AVG(mouse_distance_px) FROM dataset GROUP BY bot
""")

,bot,AVG(mouse_distance_px)
0,0,11873.544650
1,1,11392.284091


In [224]:
query (""" 
SELECT bot, AVG(idle_ratio) FROM dataset GROUP BY bot 
""")

,bot,AVG(idle_ratio)
0,0,0.496756
1,1,0.606590


In [225]:
query (""" 
SELECT bot, AVG(scroll_entropy) FROM dataset GROUP BY bot 
""")

,bot,AVG(scroll_entropy)
0,0,1.852454
1,1,1.487719


In [226]:
query (""" 
SELECT bot, AVG(digraph_latency_ms) FROM dataset GROUP BY bot 
""")

,bot,AVG(digraph_latency_ms)
0,0,108.451333
1,1,79.467448


In [227]:
query (""" 
SELECT bot, AVG(timezone_mismatch) FROM dataset GROUP BY bot 
""")

,bot,AVG(timezone_mismatch)
0,0,0.066148
1,1,0.055944


In [228]:
query (""" 
SELECT bot, AVG(webdriver_flag) FROM dataset GROUP BY bot 
""")

,bot,AVG(webdriver_flag)
0,0,0.116732
1,1,0.412587


In [229]:
query (""" 
SELECT bot, AVG(button_hover_ms) FROM dataset GROUP BY bot 
""")

,bot,AVG(button_hover_ms)
0,0,132.716829
1,1,59.440559


In [230]:
query (""" 
SELECT bot, AVG(copy_events) FROM dataset GROUP BY bot 
""")

,bot,AVG(copy_events)
0,0,1.356265
1,1,1.570629


In [231]:
query (""" 
SELECT bot, AVG(viewport_width) FROM dataset GROUP BY bot 
""")

,bot,AVG(viewport_width)
0,0,1090.474708
1,1,999.930070


## Data transformation

For machine learning and data analysis we need to convert some data types into types that machine can work with. 

Further we will be using KNN algorithm for making the predictions. KNN calculates distances, that means that we need to convert categorical columns into numeric. 

First, we separated numeric and non-numeric columns. Then used functions like get.dummies() to convert categorical columns into separate Boolean columns and astype(int) to convert Boolean values (True and False) into integers (1 and 0). 

In [232]:
# Select numeric columns (int or float)
numeric_columns = dataset.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Select non-numeric columns 
non_numeric_columns = dataset.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

print(" Numerical Columns:")
for col in numeric_columns:
    print(" -", col)

print("\n Non-Numerical (Categorical) Columns:")
for col in non_numeric_columns:
    print(" -", col)


 Numerical Columns:
 - clicks
 - median_click_interval_ms
 - click_interval_std_ms
 - mouse_path_curvature
 - mouse_distance_px
 - idle_ratio
 - scroll_entropy
 - digraph_latency_ms
 - dwell_field1_ms
 - dwell_field2_ms
 - dwell_field3_ms
 - session_duration_ms
 - pages_visited
 - timezone_mismatch
 - webdriver_flag
 - button_hover_ms
 - copy_events
 - viewport_width
 - bot

 Non-Numerical (Categorical) Columns:
 - referrer_type


In [233]:
copied_dataset = pd.get_dummies(dataset)
bool_cols = copied_dataset.select_dtypes(include="bool").columns
copied_dataset[bool_cols] = copied_dataset[bool_cols].astype(int)
print("Types: ",copied_dataset.dtypes)
copied_dataset.head()

Types:  clicks                      float64
median_click_interval_ms    float64
click_interval_std_ms       float64
mouse_path_curvature        float64
mouse_distance_px           float64
idle_ratio                  float64
scroll_entropy              float64
digraph_latency_ms          float64
dwell_field1_ms             float64
dwell_field2_ms             float64
dwell_field3_ms             float64
session_duration_ms         float64
pages_visited               float64
timezone_mismatch           float64
webdriver_flag              float64
button_hover_ms             float64
copy_events                 float64
viewport_width              float64
bot                           int64
referrer_type_ad              int64
referrer_type_direct          int64
referrer_type_search          int64
referrer_type_social          int64
dtype: object


,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,dwell_field1_ms,dwell_field2_ms,...,timezone_mismatch,webdriver_flag,button_hover_ms,copy_events,viewport_width,bot,referrer_type_ad,referrer_type_direct,referrer_type_search,referrer_type_social
0,30.0,226.0000,209.0,0.536,10735.0000,0.654000,1.726925,151.0,758.0,469.0,...,0.0,0.0,97.0,0.0,414.0,0,0,0,1,0
1,32.0,410.0000,24.0,0.451,13286.3625,0.535238,2.170000,128.0,923.0,935.0,...,0.0,0.0,136.0,1.0,414.0,1,0,0,1,0
2,31.0,530.1575,49.0,0.200,2867.0000,0.809000,1.360000,71.0,526.0,536.0,...,0.0,0.0,84.0,2.0,375.0,1,1,0,0,0
3,32.0,318.0000,75.0,0.731,8259.0000,0.525000,1.650000,86.0,786.0,725.0,...,0.0,0.0,263.0,1.0,1440.0,0,0,1,0,0
4,28.0,666.0000,517.0,0.415,51596.0000,0.535238,2.610000,101.0,705.0,429.0,...,0.0,1.0,145.0,2.0,1366.0,0,1,0,0,0


This query is needed to visualize all categorical columns that were previously converted. 

This is needed to determine whether these features might affect final result.

In [234]:
query("""
SELECT bot, AVG(referrer_type_ad), AVG(referrer_type_direct), AVG(referrer_type_search), AVG(referrer_type_social)
FROM copied_dataset GROUP BY bot
""")

,bot,AVG(referrer_type_ad),AVG(referrer_type_direct),AVG(referrer_type_search),AVG(referrer_type_social)
0,0,0.26070,0.237354,0.225681,0.276265
1,1,0.20979,0.300699,0.307692,0.181818


According to the data shown, click_interval_std_ms, differences in ( dwell_field1_ms, dwell_field2_ms, dwell_field3_ms ) are important (the data still is not enough and it might be added later)

In [235]:
# function query(sql) will help us write our SQL queries easily
def query(sql):
    return ps.sqldf(sql, globals())

In [236]:
query("""
SELECT bot,
    ROUND(AVG(session_duration_ms / NULLIF(pages_visited, 0)), 2)
        AS session_duration_per_page    
    FROM copied_dataset
GROUP BY bot;
""")

,bot,session_duration_per_page
0,0,4386.69
1,1,5292.85


After testing we find out that having session_duration_ms and pages_visited is also helpful.

In [237]:
query("""
SELECT bot,
    ROUND(AVG(click_interval_std_ms), 2)
    FROM copied_dataset
GROUP BY bot;
""")

,bot,"ROUND(AVG(click_interval_std_ms), 2)"
0,0,299.39
1,1,58.68


After testing, our assumption about feature of click_interval_std_ms was correct and it is helpful.

In [238]:
query("""
SELECT
    bot,
    ROUND (AVG((ABS(dwell_field1_ms - dwell_field2_ms) + ABS(dwell_field1_ms - dwell_field3_ms) + ABS(dwell_field2_ms - dwell_field3_ms)) / 3), 2)
        AS avg_dwell_diff
FROM copied_dataset
GROUP BY bot;
""")

,bot,avg_dwell_diff
0,0,244.05
1,1,10.55


After testing, our assumption about feature of differences between dwell features was also correct and it is helpful.

In [239]:
cleaned_dataset = query(""" 
SELECT
    bot,
    click_interval_std_ms,

    ROUND (((ABS(dwell_field1_ms - dwell_field2_ms) + ABS(dwell_field1_ms - dwell_field3_ms) + ABS(dwell_field2_ms - dwell_field3_ms) / 3)), 2)
        AS avg_dwell_diff,

    ROUND(session_duration_ms
        / NULLIF(pages_visited, 0) , 2)
        AS session_duration_per_page,

    button_hover_ms,

    mouse_path_curvature,

    webdriver_flag
    

FROM copied_dataset;
""")
cleaned_dataset

,bot,click_interval_std_ms,avg_dwell_diff,session_duration_per_page,button_hover_ms,mouse_path_curvature,webdriver_flag
0,0,209.0,870.00,1193.89,97.0,0.536,0.0
1,1,24.0,32.00,5639.50,136.0,0.451,0.0
2,1,49.0,18.67,3758.67,84.0,0.200,0.0
3,0,75.0,464.67,5161.00,263.0,0.731,0.0
4,0,517.0,390.00,3149.75,145.0,0.415,1.0
...,...,...,...,...,...,...,...
395,0,200.0,385.33,5638.01,137.0,0.439,0.0
396,0,269.0,650.67,2404.38,141.0,0.844,0.0
397,0,145.0,335.33,2109.00,122.0,0.734,0.0
398,0,160.0,1368.67,3260.80,0.0,0.856,0.0


The cleaned_dataset now is more cleaned and we can easily see the differences between robot and human

In [240]:
copied_dataset.head()

,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,dwell_field1_ms,dwell_field2_ms,...,timezone_mismatch,webdriver_flag,button_hover_ms,copy_events,viewport_width,bot,referrer_type_ad,referrer_type_direct,referrer_type_search,referrer_type_social
0,30.0,226.0000,209.0,0.536,10735.0000,0.654000,1.726925,151.0,758.0,469.0,...,0.0,0.0,97.0,0.0,414.0,0,0,0,1,0
1,32.0,410.0000,24.0,0.451,13286.3625,0.535238,2.170000,128.0,923.0,935.0,...,0.0,0.0,136.0,1.0,414.0,1,0,0,1,0
2,31.0,530.1575,49.0,0.200,2867.0000,0.809000,1.360000,71.0,526.0,536.0,...,0.0,0.0,84.0,2.0,375.0,1,1,0,0,0
3,32.0,318.0000,75.0,0.731,8259.0000,0.525000,1.650000,86.0,786.0,725.0,...,0.0,0.0,263.0,1.0,1440.0,0,0,1,0,0
4,28.0,666.0000,517.0,0.415,51596.0000,0.535238,2.610000,101.0,705.0,429.0,...,0.0,1.0,145.0,2.0,1366.0,0,1,0,0,0


In [241]:
copied_dataset = query("""
SELECT
    bot,
    clicks,
    median_click_interval_ms,
    click_interval_std_ms,
    mouse_path_curvature,
    mouse_distance_px,
    idle_ratio,
    scroll_entropy,
    digraph_latency_ms,

    ROUND(
        (dwell_field1_ms + dwell_field2_ms + dwell_field3_ms) / 3.0,
        2
    ) AS avg_dwell_time,

    session_duration_ms,
    pages_visited,
    timezone_mismatch,
    referrer_type_ad,
    referrer_type_direct,
    referrer_type_search,
    referrer_type_social,
    webdriver_flag,
    button_hover_ms,
    copy_events,
    viewport_width

FROM copied_dataset;
""")

copied_dataset.head()

,bot,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,avg_dwell_time,...,pages_visited,timezone_mismatch,referrer_type_ad,referrer_type_direct,referrer_type_search,referrer_type_social,webdriver_flag,button_hover_ms,copy_events,viewport_width
0,0,30.0,226.0000,209.0,0.536,10735.0000,0.654000,1.726925,151.0,492.33,...,9.0,0.0,0,0,1,0,0.0,97.0,0.0,414.0
1,1,32.0,410.0000,24.0,0.451,13286.3625,0.535238,2.170000,128.0,933.00,...,4.0,0.0,0,0,1,0,0.0,136.0,1.0,414.0
2,1,31.0,530.1575,49.0,0.200,2867.0000,0.809000,1.360000,71.0,528.00,...,6.0,0.0,1,0,0,0,0.0,84.0,2.0,375.0
3,0,32.0,318.0000,75.0,0.731,8259.0000,0.525000,1.650000,86.0,659.67,...,3.0,0.0,0,1,0,0,0.0,263.0,1.0,1440.0
4,0,28.0,666.0000,517.0,0.415,51596.0000,0.535238,2.610000,101.0,602.00,...,8.0,0.0,1,0,0,0,1.0,145.0,2.0,1366.0


Changing the dwell fileds features to the average of them and save it in our database  as copied_dataset

In [242]:
y = copied_dataset["bot"]
X = copied_dataset.drop(columns = "bot")

In [243]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

Split data to train and test in our model

In [244]:
y

0      0
1      1
2      1
3      0
4      0
      ..
395    0
396    0
397    0
398    0
399    0
Name: bot, Length: 400, dtype: int64

In [245]:
X

,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,avg_dwell_time,session_duration_ms,pages_visited,timezone_mismatch,referrer_type_ad,referrer_type_direct,referrer_type_search,referrer_type_social,webdriver_flag,button_hover_ms,copy_events,viewport_width
0,30.0,226.0000,209.0,0.536,10735.0000,0.654000,1.726925,151.0,492.33,10745.0000,9.0,0.0,0,0,1,0,0.0,97.0,0.0,414.0
1,32.0,410.0000,24.0,0.451,13286.3625,0.535238,2.170000,128.0,933.00,22558.0000,4.0,0.0,0,0,1,0,0.0,136.0,1.0,414.0
2,31.0,530.1575,49.0,0.200,2867.0000,0.809000,1.360000,71.0,528.00,22552.0425,6.0,0.0,1,0,0,0,0.0,84.0,2.0,375.0
3,32.0,318.0000,75.0,0.731,8259.0000,0.525000,1.650000,86.0,659.67,15483.0000,3.0,0.0,0,1,0,0,0.0,263.0,1.0,1440.0
4,28.0,666.0000,517.0,0.415,51596.0000,0.535238,2.610000,101.0,602.00,25198.0000,8.0,0.0,1,0,0,0,1.0,145.0,2.0,1366.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,26.0,530.1575,200.0,0.439,11616.0000,0.338000,2.430000,1.0,569.99,22552.0425,4.0,0.0,0,0,0,1,0.0,137.0,1.0,375.0
396,35.0,448.0000,269.0,0.844,6217.0000,0.263000,1.460000,45.0,671.67,19235.0000,8.0,1.0,0,1,0,0,0.0,141.0,3.0,1440.0
397,30.0,434.0000,145.0,0.734,9269.0000,0.535238,2.880000,101.0,543.00,16872.0000,8.0,0.0,0,0,0,1,0.0,122.0,2.0,1366.0
398,30.0,522.0000,160.0,0.856,2106.0000,0.204000,1.220000,169.0,595.33,16304.0000,5.0,0.0,0,0,1,0,0.0,0.0,1.0,1920.0


In [246]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Do the scaling part of our model based on data we are training it with 

In [247]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


making the KNN classifiers based on our data 

In [248]:
y_pred = model.predict(X_test_scaled)
base_mae = mean_absolute_error(y_test, y_pred)
base_rmse = sqrt(mean_squared_error(y_test, y_pred))
base_accuracy =  accuracy_score(y_test, y_pred)
print("MAE",base_mae)
print("RMSE",base_rmse)
print("Accuracy",base_accuracy)


MAE 0.0625
RMSE 0.25
Accuracy 0.9375


Calculate and Check MAE, RMSE, and Accuracy for our data with all the features included

In [249]:
cleaned_X_train = X_train
cleaned_X_test = X_test
for feature in X:
    X_train_dropped = cleaned_X_train.drop(columns = [feature])
    X_test_dropped = cleaned_X_test.drop(columns = [feature])
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_dropped)
    X_test_scaled = scaler.transform(X_test_dropped)
    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = sqrt(mean_squared_error(y_test, y_pred))
    if rmse - base_rmse <= 0:
        print("Removed Feature: ", feature)
        cleaned_X_train = cleaned_X_train.drop(columns = [feature])
        cleaned_X_test = cleaned_X_test.drop(columns = [feature])

Removed Feature:  session_duration_ms
Removed Feature:  timezone_mismatch
Removed Feature:  referrer_type_direct
Removed Feature:  referrer_type_social
Removed Feature:  webdriver_flag
Removed Feature:  copy_events
Removed Feature:  viewport_width


Removing the data that did not have any effect or made our data rmse worse

In [250]:
X_train

,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,avg_dwell_time,session_duration_ms,pages_visited,timezone_mismatch,referrer_type_ad,referrer_type_direct,referrer_type_search,referrer_type_social,webdriver_flag,button_hover_ms,copy_events,viewport_width
3,32.0000,318.0000,75.0000,0.731,8259.0,0.525000,1.650000,86.0000,659.67,15483.0,3.0000,0.0,0,1,0,0,0.0,263.000,1.0,1440.0
18,29.0000,530.1575,355.0000,0.585,9846.0,0.411000,1.726925,134.0000,778.00,35412.0,5.0000,0.0,0,1,0,0,0.0,114.015,1.0,1366.0
202,32.0000,358.0000,26.0000,0.222,34721.0,0.660000,1.710000,106.0000,235.00,18581.0,5.5225,0.0,0,0,1,0,1.0,123.000,2.0,1440.0
250,32.0000,483.0000,459.0000,0.638,9585.0,0.276000,2.680000,45.0000,602.33,18627.0,7.0000,0.0,0,0,0,1,0.0,25.000,0.0,1366.0
274,24.0000,816.0000,127.0000,0.807,3338.0,0.392000,1.320000,158.0000,456.99,28636.0,8.0000,0.0,1,0,0,0,0.0,91.000,1.0,375.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,33.0000,317.0000,26.0000,0.509,3304.0,0.545000,0.830000,103.0000,217.33,13205.0,3.0000,0.0,0,0,1,0,0.0,0.000,3.0,375.0
106,31.0000,538.0000,226.4525,0.188,28345.0,0.535238,1.726925,68.0000,294.00,26665.0,5.0000,0.0,0,1,0,0,0.0,155.000,3.0,414.0
270,28.0000,761.0000,236.0000,0.553,4358.0,0.275000,1.490000,102.9225,401.33,25692.0,4.0000,0.0,0,1,0,0,0.0,259.000,1.0,1366.0
348,27.8875,598.0000,285.0000,0.807,3182.0,0.588000,2.220000,186.0000,659.67,37934.0,8.0000,0.0,0,0,1,0,1.0,217.000,1.0,1920.0


In [251]:
cleaned_X_train

,clicks,median_click_interval_ms,click_interval_std_ms,mouse_path_curvature,mouse_distance_px,idle_ratio,scroll_entropy,digraph_latency_ms,avg_dwell_time,pages_visited,referrer_type_ad,referrer_type_search,button_hover_ms
3,32.0000,318.0000,75.0000,0.731,8259.0,0.525000,1.650000,86.0000,659.67,3.0000,0,0,263.000
18,29.0000,530.1575,355.0000,0.585,9846.0,0.411000,1.726925,134.0000,778.00,5.0000,0,0,114.015
202,32.0000,358.0000,26.0000,0.222,34721.0,0.660000,1.710000,106.0000,235.00,5.5225,0,1,123.000
250,32.0000,483.0000,459.0000,0.638,9585.0,0.276000,2.680000,45.0000,602.33,7.0000,0,0,25.000
274,24.0000,816.0000,127.0000,0.807,3338.0,0.392000,1.320000,158.0000,456.99,8.0000,1,0,91.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,33.0000,317.0000,26.0000,0.509,3304.0,0.545000,0.830000,103.0000,217.33,3.0000,0,1,0.000
106,31.0000,538.0000,226.4525,0.188,28345.0,0.535238,1.726925,68.0000,294.00,5.0000,0,0,155.000
270,28.0000,761.0000,236.0000,0.553,4358.0,0.275000,1.490000,102.9225,401.33,4.0000,0,0,259.000
348,27.8875,598.0000,285.0000,0.807,3182.0,0.588000,2.220000,186.0000,659.67,8.0000,0,1,217.000


In [252]:
X_train_scaled = scaler.fit_transform(cleaned_X_train)
X_test_scaled = scaler.transform(cleaned_X_test)
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

Training the model once more

In [253]:
best_mae = mean_absolute_error(y_test, y_pred)
best_rmse = sqrt(mean_squared_error(y_test, y_pred))
best_accuracy =  accuracy_score(y_test, y_pred)
print("MAE",best_mae)
print("RMSE",round(best_rmse,3))
print("Accuracy", best_accuracy)

MAE 0.025
RMSE 0.158
Accuracy 0.975


Getting the new mae, rmse, and accuracy based on our new dataset (with reduced features)

In [254]:
dump(model, "knn_model.joblib")
dump(scaler, "scaler.joblib")
print("Model and scaler saved successfully!")

Model and scaler saved successfully!


In [255]:
def dataset_creation (features, newData):
    newData = newData.copy()
    newData["avg_dwell_time"] = ( newData["dwell_field1_ms"] + newData["dwell_field2_ms"] + newData["dwell_field3_ms"] ) / 3
    newData.drop(columns = ["dwell_field1_ms", "dwell_field2_ms", "dwell_field3_ms"])
    newData = newData[features]
    return newData